In [5]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from datetime import datetime, timedelta
import datajoint as dj

In [7]:
dj.config.load("dj_local_conf_worker.json")
dj.conn()

[2026-04-20 12:16:05,518][INFO]: DataJoint is configured from c:\Users\Organoid PC\Documents\GitHub\utah_organoids\dj_local_conf_worker.json


DataJoint connection (connected) utah-worker@db.datajoint.com:3306

In [4]:
from workflow.pipeline import frame, culture, ephys, analysis, mua, probe, ephys_sorter

Module stfio is not installed. Make sure .abf files are converted to .pkl in Python 2.


Notebook for full frame analysis

In [8]:
# define organoid ids to insert into the database
organoid_ids = ["O21", "O22", "O23", "O24", "O25", "O26", "O27", "O28", "O29", "O30", "O31", "O32"]

In [9]:
# add function to define boundaries/ keys for frame analysis
def get_frame_key(organoid_id):
    frame_param_idx = 6
    
    # get experiment key
    drug_name = "Control"
    exp_key = (culture.Experiment & f"organoid_id = '{organoid_id}'" & f"drug_name = '{drug_name}'").fetch1("KEY")

    # get boundaries for analysis
    boundary_start = (culture.Experiment & exp_key).fetch1("experiment_start_time") + timedelta(hours=1)
    boundary_end = boundary_start + timedelta(days=2)

    # get frame key
    frame_key = {
        **exp_key,
        "start_boundary": boundary_start,
        "end_boundary": boundary_end,
        "frame_param_idx": frame_param_idx
    }
    return frame_key

In [10]:
def populate_frame_session(frame_key):

    # insert frame session
    frame.FrameSession.insert1(frame_key, skip_duplicates=True)

    # populate frame analysis
    frame.FrameAnalysis.populate(frame_key)

    return

def get_key(active_frame_key):
    insertion_number = 2

    return {
        "organoid_id": active_frame_key["organoid_id"],
        "experiment_start_time": active_frame_key["experiment_start_time"],
        "start_time": active_frame_key["frame_start"],
        "end_time": active_frame_key["frame_end"],
        "insertion_number": insertion_number
    }

def populate_ephys_session(key):
    session_type = "both"
    probe = "Q983"
    used_electrodes = []

    # find port id from organoid_id
    port_id = set((ephys.EphysSessionProbe & f"organoid_id = '{key['organoid_id']}'").fetch("port_id"))
    if len(port_id) != 1:
        raise ValueError(
            f"Unable to determine port_id for organoid {key['organoid_id']}. Found port_ids: {port_id}"
        )
    port_id = port_id.pop()

    # get ephys key
    ephys_key = {
        **key,
        "session_type": session_type
    }

    # get probe key
    probe_key = {
        **key,
        "probe": probe,
        "port_id": port_id,
        "used_electrodes": used_electrodes
    }

    # insert ephys session and probe
    ephys.EphysSession.insert1(ephys_key, skip_duplicates=True)
    ephys.EphysSessionProbe.insert1(probe_key, skip_duplicates=True)
    ephys.EphysSessionInfo.populate(key)

    return

def populate_burst_session(key):
    burst_param_idx = 1

    # get burst key
    burst_key = {
        **key,
        "burst_param_idx": burst_param_idx
    } 

    # insert burst session
    mua.BurstSession.insert1(burst_key, skip_duplicates=True)
    mua.PopulationBursts.populate(key)

    return

def populate_spectral_session(key):
    param_idx = 0

    # populate parent key
    ephys.LFP.populate(key)

    # get trace keys
    trace_keys = [
        {**trace_key, "param_idx": param_idx} 
        for trace_key in (ephys.LFP.Trace & key).fetch("KEY")
        ]
    
    # populate spectral analysis
    analysis.LFPSpectrogram.populate(trace_keys)
    analysis.STTFA.populate(trace_keys)

def populate_coherence_session(key):
    
    # populate coherence
    analysis.Coherence.populate(key)

    return

def populate_fooof_session(key):
    fooof_param_idx = 3
    fbosc_param_idx = 3
    start_freq = 2
    end_freq = 300
    analysis_electrodes = list(range((culture.OrganoidImplantationImage & f"organoid_id = '{key['organoid_id']}'").fetch1("num_electrodes_inside"))) 

    # get frame key
    fooof_key = {
        **key,
        "fooof_param_idx": fooof_param_idx,
        "fbosc_param_idx": fbosc_param_idx,
        "start_freq": start_freq,
        "end_freq": end_freq,
        "analysis_electrodes": analysis_electrodes
    }

    # insert manual fooof session
    analysis.FOOOFandFBOSCSession.insert1(fooof_key, skip_duplicates=True)

    # populate fooof analysis
    fetch_key = fooof_key.copy()
    fetch_key.pop("analysis_electrodes")

    populate_key = (analysis.FOOOFandFBOSCSession & fetch_key).fetch1("KEY")
    analysis.FOOOFAnalysis.populate(populate_key)

    return

In [27]:
# populate all sessions for each organoid
for organoid_id in organoid_ids:

    print(f"Populating data for organoid {organoid_id}...")
    start_time = datetime.now()

    # get frame key
    frame_key = get_frame_key(organoid_id)
    
    # populate frame session
    populate_frame_session(frame_key)
    active_frame_keys = (frame.FrameAnalysis.ActiveTimeFrames & frame_key).fetch("KEY")

    # loop through active frame keys and populate sessions
    for i, active_frame_key in enumerate(active_frame_keys):

        print(f"       Frame {i+1}/{len(active_frame_keys)}")

        # get key for other sessions
        key = get_key(active_frame_key)
        print(f"        key: {key}")

        # populate ephys session
        populate_ephys_session(key)

        # populate burst session
        populate_burst_session(key)

        # populate spectral session
        populate_spectral_session(key)

        # populate coherence session
        populate_coherence_session(key)

        # populate fooof session
        populate_fooof_session(key)
    
    end_time = datetime.now()
    print(f"Finished populating data for organoid {organoid_id}. Time taken: {end_time - start_time}")

Populating data for organoid O21...


[2026-04-21 12:29:11,894][WARNING]: Reconnecting to MySQL server.


       Frame 1/1
        key: {'organoid_id': 'O21', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'start_time': datetime.datetime(2024, 7, 3, 11, 51, 5), 'end_time': datetime.datetime(2024, 7, 3, 11, 56, 4), 'insertion_number': 2}
Finished populating data for organoid O21. Time taken: 0:00:23.554663
Populating data for organoid O22...
       Frame 1/8
        key: {'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'start_time': datetime.datetime(2024, 7, 1, 16, 2, 4), 'end_time': datetime.datetime(2024, 7, 1, 16, 7, 3), 'insertion_number': 2}
       Frame 2/8
        key: {'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'start_time': datetime.datetime(2024, 7, 2, 23, 3, 5), 'end_time': datetime.datetime(2024, 7, 2, 23, 8, 4), 'insertion_number': 2}
       Frame 3/8
        key: {'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'start_time': datetime.datetime

[2026-04-21 12:30:29,999][INFO]: Populating ephys.EphysSessionInfo for <{'organoid_id': 'O30', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 8, 59, 14), 'end_time': datetime.datetime(2024, 10, 31, 9, 4, 13)}>
[2026-04-21 12:30:31,689][INFO]: Populating ephys.LFP for <{'organoid_id': 'O30', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 8, 59, 14), 'end_time': datetime.datetime(2024, 10, 31, 9, 4, 13)}>
[2026-04-21 12:31:52,138][INFO]: Populating ephys.LFP for <{'organoid_id': 'O30', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 8, 59, 14), 'end_time': datetime.datetime(2024, 10, 31, 9, 4, 13)}>
c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a pa

Finished populating data for organoid O30. Time taken: 0:06:58.721034
Populating data for organoid O31...
       Frame 1/8
        key: {'organoid_id': 'O31', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'start_time': datetime.datetime(2024, 10, 30, 15, 23, 37), 'end_time': datetime.datetime(2024, 10, 30, 15, 28, 36), 'insertion_number': 2}


...

       Frame 7/8
        key: {'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'start_time': datetime.datetime(2024, 10, 31, 6, 28, 14), 'end_time': datetime.datetime(2024, 10, 31, 6, 33, 13), 'insertion_number': 2}


[2026-04-21 14:16:26,746][INFO]: Populating ephys.EphysSessionInfo for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 6, 28, 14), 'end_time': datetime.datetime(2024, 10, 31, 6, 33, 13)}>
[2026-04-21 14:16:28,743][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 6, 28, 14), 'end_time': datetime.datetime(2024, 10, 31, 6, 33, 13)}>
[2026-04-21 14:17:51,424][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 6, 28, 14), 'end_time': datetime.datetime(2024, 10, 31, 6, 33, 13)}>
c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a

       Frame 8/8
        key: {'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'start_time': datetime.datetime(2024, 10, 31, 8, 58, 14), 'end_time': datetime.datetime(2024, 10, 31, 9, 3, 13), 'insertion_number': 2}


[2026-04-21 14:23:31,054][INFO]: Populating ephys.EphysSessionInfo for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 8, 58, 14), 'end_time': datetime.datetime(2024, 10, 31, 9, 3, 13)}>
[2026-04-21 14:23:33,164][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 8, 58, 14), 'end_time': datetime.datetime(2024, 10, 31, 9, 3, 13)}>
[2026-04-21 14:24:52,528][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 10, 31, 8, 58, 14), 'end_time': datetime.datetime(2024, 10, 31, 9, 3, 13)}>
c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a pa

Finished populating data for organoid O32. Time taken: 0:56:18.290642


In [8]:
for organoid_id in organoid_ids:

    print(f"Populating FOOOF data for organoid {organoid_id}...")
    
    frame_key = get_frame_key(organoid_id)
    active_frame_keys = (frame.FrameAnalysis.ActiveTimeFrames & frame_key).fetch("KEY")

    for active_frame_key in active_frame_keys:

        key = get_key(active_frame_key)

        populate_fooof_session(key)

Populating FOOOF data for organoid O09...
Populating FOOOF data for organoid O10...
Populating FOOOF data for organoid O11...
Populating FOOOF data for organoid O12...
Populating FOOOF data for organoid O17...
Populating FOOOF data for organoid O18...
Populating FOOOF data for organoid O19...
Populating FOOOF data for organoid O20...
Populating FOOOF data for organoid O13...
Populating FOOOF data for organoid O14...
Populating FOOOF data for organoid O15...
Populating FOOOF data for organoid O16...


In [20]:
frame.FrameAnalysis.ActiveTimeFrames & frame_key

organoid_id e.g. O17,experiment_start_time,start_boundary Start datetime for analysis,end_boundary End datetime for analysis,frame_param_idx Reference to TimeFrameParamset,frame_start Start of active time frame,frame_end End of active time frame,frame_firing_rate Firing rates for each frame
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 19:43:49,2023-05-03 19:48:48,0.01
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 20:32:49,2023-05-03 20:37:48,0.0533334
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 22:11:49,2023-05-03 22:16:48,0.01
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 23:00:49,2023-05-03 23:05:48,0.0233334
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-04 12:48:52,2023-05-04 12:53:51,0.01
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-04 15:17:52,2023-05-04 15:22:51,0.00666668
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-04 17:45:52,2023-05-04 17:50:51,0.0133334
O10,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-05 17:04:52,2023-05-05 17:09:51,0.0466668


In [28]:
# add function to define boundaries/ keys for frame analysis
def get_frame_key(organoid_id):
    frame_param_idx = 6
    
    # get experiment key
    drug_name = "Control"
    exp_key = (culture.Experiment & f"organoid_id = '{organoid_id}'" & f"drug_name = '{drug_name}'").fetch1("KEY")

    # get boundaries for analysis
    boundary_end = (culture.Experiment & exp_key).fetch1("experiment_end_time") - timedelta(hours=1)
    boundary_start = boundary_end - timedelta(days=2)

    # get frame key
    frame_key = {
        **exp_key,
        "start_boundary": boundary_start,
        "end_boundary": boundary_end,
        "frame_param_idx": frame_param_idx
    }
    return frame_key

In [10]:
for organoid_id in organoid_ids:

    print(f"Populating FOOOF data for organoid {organoid_id}...")
    
    frame_key = get_frame_key(organoid_id)
    active_frame_keys = (frame.FrameAnalysis.ActiveTimeFrames & frame_key).fetch("KEY")

    for active_frame_key in active_frame_keys:

        key = get_key(active_frame_key)

        populate_fooof_session(key)

Populating FOOOF data for organoid O09...
Populating FOOOF data for organoid O10...
Populating FOOOF data for organoid O11...
Populating FOOOF data for organoid O12...
Populating FOOOF data for organoid O17...
Populating FOOOF data for organoid O18...
Populating FOOOF data for organoid O19...
Populating FOOOF data for organoid O20...
Populating FOOOF data for organoid O13...
Populating FOOOF data for organoid O14...
Populating FOOOF data for organoid O15...
Populating FOOOF data for organoid O16...


In [29]:
# populate all sessions for each organoid
for organoid_id in organoid_ids:

    print(f"Populating data for organoid {organoid_id}...")
    start_time = datetime.now()

    # get frame key
    frame_key = get_frame_key(organoid_id)
    
    # populate frame session
    populate_frame_session(frame_key)
    active_frame_keys = (frame.FrameAnalysis.ActiveTimeFrames & frame_key).fetch("KEY")

    # loop through active frame keys and populate sessions
    for i, active_frame_key in enumerate(active_frame_keys):

        print(f"       Frame {i+1}/{len(active_frame_keys)}")

        # get key for other sessions
        key = get_key(active_frame_key)
        print(f"        key: {key}")

        # populate ephys session
        populate_ephys_session(key)

        # populate burst session
        populate_burst_session(key)

        # populate spectral session
        populate_spectral_session(key)

        # populate coherence session
        populate_coherence_session(key)

        # populate fooof session
        populate_fooof_session(key)
    
    end_time = datetime.now()
    print(f"Finished populating data for organoid {organoid_id}. Time taken: {end_time - start_time}")

Populating data for organoid O21...


[2026-04-21 15:50:40,295][WARNING]: Reconnecting to MySQL server.


Finished populating data for organoid O21. Time taken: 0:00:25.145545
Populating data for organoid O22...
       Frame 1/8
        key: {'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'start_time': datetime.datetime(2024, 7, 7, 13, 32, 6), 'end_time': datetime.datetime(2024, 7, 7, 13, 37, 5), 'insertion_number': 2}


[2026-04-21 15:50:53,742][INFO]: Populating ephys.EphysSessionInfo for <{'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 7, 7, 13, 32, 6), 'end_time': datetime.datetime(2024, 7, 7, 13, 37, 5)}>
C:\Users\Organoid PC\Documents\GitHub\utah_organoids\src\workflow\pipeline\mua.py:722: RuntimeWarning:

invalid value encountered in divide

[2026-04-21 15:50:55,429][INFO]: Populating ephys.LFP for <{'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 7, 7, 13, 32, 6), 'end_time': datetime.datetime(2024, 7, 7, 13, 37, 5)}>
[2026-04-21 15:52:16,407][INFO]: Populating ephys.LFP for <{'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 7, 7, 13, 32, 6), 'end_time': datetime.datetime(2024, 7, 7, 13, 37, 5)}>
c:\Use

       Frame 2/8
        key: {'organoid_id': 'O22', 'experiment_start_time': datetime.datetime(2024, 7, 1, 13, 53), 'start_time': datetime.datetime(2024, 7, 7, 14, 48, 6), 'end_time': datetime.datetime(2024, 7, 7, 14, 53, 5), 'insertion_number': 2}


...

       Frame 7/8
        key: {'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'start_time': datetime.datetime(2024, 11, 11, 6, 54, 17), 'end_time': datetime.datetime(2024, 11, 11, 6, 59, 16), 'insertion_number': 2}


[2026-04-21 22:22:19,172][INFO]: Populating ephys.EphysSessionInfo for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 11, 11, 6, 54, 17), 'end_time': datetime.datetime(2024, 11, 11, 6, 59, 16)}>
C:\Users\Organoid PC\Documents\GitHub\utah_organoids\src\workflow\pipeline\mua.py:722: RuntimeWarning:

invalid value encountered in divide

[2026-04-21 22:22:20,825][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 11, 11, 6, 54, 17), 'end_time': datetime.datetime(2024, 11, 11, 6, 59, 16)}>
[2026-04-21 22:23:40,114][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 11, 11, 6, 54, 17), 'end_time': datetime.datetime(2024, 11, 11, 6

       Frame 8/8
        key: {'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'start_time': datetime.datetime(2024, 11, 11, 8, 21, 17), 'end_time': datetime.datetime(2024, 11, 11, 8, 26, 16), 'insertion_number': 2}


[2026-04-21 22:28:54,885][INFO]: Populating ephys.EphysSessionInfo for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 11, 11, 8, 21, 17), 'end_time': datetime.datetime(2024, 11, 11, 8, 26, 16)}>
[2026-04-21 22:28:56,600][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 11, 11, 8, 21, 17), 'end_time': datetime.datetime(2024, 11, 11, 8, 26, 16)}>
[2026-04-21 22:30:16,429][INFO]: Populating ephys.LFP for <{'organoid_id': 'O32', 'experiment_start_time': datetime.datetime(2024, 10, 30, 14, 10), 'insertion_number': 2, 'start_time': datetime.datetime(2024, 11, 11, 8, 21, 17), 'end_time': datetime.datetime(2024, 11, 11, 8, 26, 16)}>
c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a

Finished populating data for organoid O32. Time taken: 0:54:13.569480


In [10]:
param_idx = 0

# get trace keys
trace_keys = [
    {**trace_key, "param_idx": param_idx} 
    for trace_key in (ephys.LFP.Trace & key).fetch("KEY")
    ]


[2026-03-30 19:38:01,273][WARNING]: Reconnecting to MySQL server.


In [13]:
spike_indices, start_times = (mua.MUASpikes.Channel & 
                                            f"organoid_id='{key['organoid_id']}'" &
                                            f"start_time BETWEEN '{key['start_time']}' AND '{key['end_time']}'" &
                                            f"channel_idx = '0'"
                                            ).fetch('spike_indices', 'start_time')

In [16]:
np.concatenate(spike_indices)

array([], dtype=int64)

In [ ]:
# populate spectral analysis
analysis.STTFA.populate(trace_keys)


{'success_count': 0, 'error_list': []}

In [35]:
(mua.PopulationBursts & key).fetch1("weighted_sttc")

array([nan, nan])

In [11]:
key = {
    **(ephys.EphysSession & "organoid_id = 'O10'" & "insertion_number = 2").fetch1("KEY"),
    "burst_param_idx": 1
       }

In [8]:
from element_array_ephys.ephys_no_curation import map_channel_to_electrode, get_probe_type
from scipy.signal import find_peaks
import bottleneck as bn
from scipy.ndimage import gaussian_filter1d

In [12]:
import neo
import quantities as pq
from elephant.spike_train_correlation import spike_time_tiling_coefficient

# define parameters
fs = 20000 # sampling frequency in Hz — Intan acquisition rate; hardcoded since MUASpikes.spike_indices are stored as raw sample indices at this rate and changing acquisition systems would require repopulating MUASpikes
burst_extract_dur = np.timedelta64(1, 's') # time for extracting burst spike array (+ and - from peak)
burst_bound_thresh = 0.1 # threshold for defining burst bounds (percentage of peak height)

# Fetch MUA parameters within the frame
spike_indices, start_times, channel_ids = (mua.MUASpikes.Channel & 
                                            f"organoid_id='{key['organoid_id']}'" &
                                            f"start_time BETWEEN '{key['start_time']}' AND '{key['end_time']}'"
                                            ).fetch('spike_indices', 'start_time', 'channel_idx')

# check if we have spike indices for all times
if len(np.unique(start_times)) < np.timedelta64(key['end_time'] - key['start_time'], 'm') / np.timedelta64(1,'m'):
    raise ValueError(f"Not all time windows have MUA spike data for {key} - cannot perform burst detection")

# convert channel ids to electrode indices
probe_type = get_probe_type(key)
electrode_ids = map_channel_to_electrode(probe_type, input_indices=channel_ids)

# get array of all spike times (relative to frame start)
start_ms = (start_times - key['start_time']).astype('timedelta64[ms]') / np.timedelta64(1, 'ms') # ms from frame start
rel_spike_times_ms = spike_indices / fs / (np.timedelta64(1,'ms')/np.timedelta64(1,'s')) 
spike_times_ms = rel_spike_times_ms + start_ms

# fetch electrode count from implantation image (source of truth)
img_query = culture.OrganoidImplantationImage & {"organoid_id": key["organoid_id"]}
if not img_query:
    raise ValueError(f"No OrganoidImplantationImage entry found for organoid_id='{key['organoid_id']}' - insert a row before running this computation")
if len(img_query) > 1:
    raise ValueError(f"Multiple OrganoidImplantationImage entries found for organoid_id='{key['organoid_id']}' - expected exactly one")
num_elec_inside = img_query.fetch1("num_electrodes_inside")
if num_elec_inside is None:
    raise ValueError(f"num_electrodes_inside is not set in OrganoidImplantationImage for organoid_id='{key['organoid_id']}'")
elec_bool = (electrode_ids >= 0) & (electrode_ids < num_elec_inside)

# create population spike time series (1 ms bins)
time_bins = np.arange(0, np.timedelta64(key['end_time'] - key['start_time'], 'ms') / np.timedelta64(1, 'ms') + 1) # 1 ms bins
population_spike_series, _ = np.histogram(np.hstack(spike_times_ms[elec_bool]), bins=time_bins)

# convert spike series to firing rate
population_firing_rate = population_spike_series * 1000 # convert to spikes per second

# smooth firing rate with Gaussian and Boxcar kernels
# fetch burst detection parameters
gaus_len_ms, boxcar_len_ms, detection_threshold, min_distance_ms = (mua.BurstDetectionParamset & key).fetch1(
    'gaus_len_ms', 'boxcar_len_ms', 'detection_threshold', 'min_distance_ms'
)
# boxcar kernel
population_firing_rate = bn.move_mean(population_firing_rate, window=boxcar_len_ms, min_count=1)

# Gaussian kernel 
truncate = 4
population_firing_rate = gaussian_filter1d(population_firing_rate, sigma=gaus_len_ms, truncate=truncate, mode="reflect")

# detect spike bursts
min_height = detection_threshold * np.std(population_firing_rate)

# find peaks
burst_indices, properties = find_peaks(population_firing_rate, height=min_height, distance=min_distance_ms)
burst_peak_heights = properties['peak_heights']

# find burst bounds (start and end indices where firing rate >= 10% of peak height)

# define burst extraction parameters
num_burst_samples = int(burst_extract_dur / np.timedelta64(1,'ms')) # number of samples to extract from burst peak (+ and -)

# remove boundary bursts (will raise an error when extracting burst windows)
boundary_bool = (num_burst_samples <= burst_indices) & (burst_indices <= (len(population_firing_rate)-num_burst_samples))
burst_indices = burst_indices[boundary_bool]
burst_peak_heights = burst_peak_heights[boundary_bool]

# find burst windows and create spike array
burst_windows = []
burst_spike_array = np.zeros((len(burst_indices), num_elec_inside, 2*num_burst_samples), dtype=bool)
for burst_idx, (index, height) in enumerate(zip(burst_indices, burst_peak_heights)):

    # extract burst waveform
    waveform = population_firing_rate[index-num_burst_samples : index+num_burst_samples]

    # find burst specific window threshold
    window_thresh = burst_bound_thresh * height
    window = np.array([0, 0])

    # find number of indices adjacent to the burst peak are over the burst threshold
    i = 1
    while (waveform[num_burst_samples-i] >= window_thresh) & (num_burst_samples-i > 0): # make sure it doesn't exceed the number of extracted samples
        window[0] -= 1 # indices before burst peak
        i += 1
    i = 1
    while (waveform[num_burst_samples+i] >= window_thresh) & (num_burst_samples+i < len(waveform)-1):
        window[1] += 1 # indices after burst peak
        i += 1        
    
    burst_windows.append(window)

    # fill in spike array for each electrode
    for elec_idx in range(num_elec_inside):

        # get spike times for electrode
        elec_spike_times = np.hstack(spike_times_ms[electrode_ids == elec_idx])

        # find spikes within burst window
        burst_spike_times = elec_spike_times[((index-num_burst_samples) <= elec_spike_times) & (elec_spike_times < (index+num_burst_samples))]

        # convert to indices within burst spike array
        burst_spike_indices = (burst_spike_times - (index-num_burst_samples)).astype(int)
        burst_spike_array[burst_idx, elec_idx, burst_spike_indices] = True
burst_bounds = np.array(burst_windows)

# determine spike time tiling coefficient (functional connectivity) for each burst (weighted by spikes per pair)
sttc_array = np.zeros((len(burst_indices), num_elec_inside, num_elec_inside))
weight_array = np.zeros((len(burst_indices), num_elec_inside, num_elec_inside))

dt = 5 # time window for STTC in ms
t_stop = 2*num_burst_samples # total time window for spike trains in ms
for b_idx in range(len(burst_indices)):
    for i in range(num_elec_inside):
        for j in range(i+1, num_elec_inside):
            
            # define spike times for each electrode within burst (in ms)
            spike_times_i = np.where(burst_spike_array[b_idx, i, :])[0]
            spike_times_j = np.where(burst_spike_array[b_idx, j, :])[0]

            # skip if either electrode has no spikes in burst
            if len(spike_times_i) == 0 or len(spike_times_j) == 0:
                continue


            # convert to spike trains (neo)
            spiketrain_A = neo.SpikeTrain(spike_times_i, units='ms', t_stop=t_stop)
            spiketrain_B = neo.SpikeTrain(spike_times_j, units='ms', t_stop=t_stop)

            # calculate STTC
            sttc = spike_time_tiling_coefficient(spiketrain_A, spiketrain_B, dt=dt*pq.ms)   
            sttc_array[b_idx, i, j] = sttc

            # calculate weight (number of spike pairs)
            weight_array[b_idx, i, j] = len(spike_times_i) * len(spike_times_j)

# determine weighted average STTC for each burst; NaN for bursts with no spike pairs
total_weight = weight_array.sum(axis=(1, 2))
weighted_sttc = np.where(
    total_weight > 0,
    (sttc_array * weight_array).sum(axis=(1, 2)) / total_weight,
    np.nan,
)

C:\Users\Organoid PC\AppData\Local\Temp\ipykernel_13324\4087171343.py:148: RuntimeWarning:

invalid value encountered in divide



In [27]:
np.where(
    total_weight > 0,
    (sttc_array * weight_array).sum(axis=(1, 2)) ,
    np.nan,
)

array([nan, nan])

In [17]:
frame.FrameAnalysis.ActiveTimeFrames & "organoid_id = 'O09'" & "frame_param_idx = 6" & f"start_boundary < '{datetime(2023, 5, 10)}'"

organoid_id e.g. O17,experiment_start_time,start_boundary Start datetime for analysis,end_boundary End datetime for analysis,frame_param_idx Reference to TimeFrameParamset,frame_start Start of active time frame,frame_end End of active time frame,frame_firing_rate Firing rates for each frame
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 19:40:49,2023-05-03 19:45:48,1.88
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 20:23:49,2023-05-03 20:28:48,1.39
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 20:38:49,2023-05-03 20:43:48,6.75001
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-03 22:54:49,2023-05-03 22:59:48,5.79667
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-04 04:11:52,2023-05-04 04:16:51,14.9833
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-04 05:16:52,2023-05-04 05:21:51,11.2667
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-04 16:17:52,2023-05-04 16:22:51,19.4067
O09,2023-05-03 17:33:00,2023-05-03 18:33:00,2023-05-05 18:33:00,6,2023-05-05 17:28:52,2023-05-05 17:33:51,2.55333


In [16]:
start_time = datetime.now()

In [27]:
framekey9 = (frame.FrameSession & "organoid_id = 'O09'" & "frame_param_idx = 6" & f"start_boundary < '{datetime(2023, 5, 10)}'").fetch1("KEY")
active_frame_keys = (frame.FrameAnalysis.ActiveTimeFrames & framekey9).fetch("KEY")

In [28]:
example_keys = [get_key(active_frame_key) for active_frame_key in active_frame_keys]

In [10]:
fooof_keys = (analysis.FOOOFandFBOSCSession * analysis.SpectrogramParameters & analysis.LFPSpectrogram & example_keys).fetch("KEY")

In [11]:
analysis.FOOOFAnalysis.populate(fooof_keys)

c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a path object that I don't think is part of a bar chart. Ignoring.

c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a path object that I don't think is part of a bar chart. Ignoring.

c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a path object that I don't think is part of a bar chart. Ignoring.

c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a path object that I don't think is part of a bar chart. Ignoring.

c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a path object that I don't think is part of a bar chart. Ignoring.

c:\Users\Organoid PC

{'success_count': 8, 'error_list': []}

In [34]:
key = fooof_keys[0]

In [33]:
from specparam import SpectralModel
from scipy.interpolate import interp1d
import plotly.tools as tls
import plotly.io as pio
from scipy.stats import chi2

In [14]:
def interpolate_spectrum(frequency, spectrum, notch_freqs):
    # create mask for frequencies to remove
    mask = np.ones_like(frequency, dtype=bool)
    for notch_freq in notch_freqs:
        freq_mask = (notch_freq - 5 <= frequency) & (frequency <= notch_freq + 5)
        mask = mask & (~freq_mask)
    # interpolate
    interp_func = interp1d(frequency[mask], spectrum[mask], kind='linear', fill_value="extrapolate")
    interp_spectrum = interp_func(frequency)
    
    return interp_spectrum

In [35]:
# fetch electrodes to analyze
analysis_electrodes = (analysis.FOOOFandFBOSCSession & key).fetch1("analysis_electrodes")

# fetch time and frequency information
time, frequency = np.array((analysis.LFPSpectrogram.ChannelSpectrogram & key).fetch("time", "frequency"))[:,0]

# fetch lfp spectrograms (averaged across electrodes)
if len(analysis_electrodes) > 0:
    spectrograms = (analysis.LFPSpectrogram.ChannelSpectrogram & key & f"electrode IN {tuple(analysis_electrodes)}").fetch("spectrogram")
else:
    spectrograms = (analysis.LFPSpectrogram.ChannelSpectrogram & key).fetch("spectrogram")
mean_spectrum = np.mean(np.stack(spectrograms, axis=-1), axis=2)  # shape: (frequency, time)

# fetch fooof parameters
peak_width_limits, max_n_peaks, min_peak_height, peak_threshold, aperiodic_mode = (analysis.FOOOFParamset & key).fetch1(
    "peak_width_limits", "max_n_peaks", "min_peak_height", "peak_threshold", "aperiodic_mode"
)

# fetch fooof session parameters
start_freq, end_freq = (analysis.FOOOFandFBOSCSession & key).fetch1(
    "start_freq", "end_freq"
)
bounded_frequency = frequency[(start_freq <= frequency) & (frequency <= end_freq)]

# get frequency band information (mask if frequency is within band)
frequency_band_masks = {
    band['band_name']: (band['lower_freq'] <= bounded_frequency) & (bounded_frequency <= band['upper_freq'])
    for band in analysis.SpectralBand.fetch(as_dict=True, order_by='lower_freq')
    } 

# initialize model
fm = SpectralModel(
    peak_width_limits=peak_width_limits,
    max_n_peaks=max_n_peaks,
    min_peak_height=min_peak_height,
    peak_threshold=peak_threshold,
    aperiodic_mode=aperiodic_mode,
    verbose=False
)

# process summary fooof fit over all time bins
notch_freqs = np.arange(60, frequency.max(), 60)
interp_spectrum = interpolate_spectrum(frequency, np.mean(mean_spectrum, axis=1), notch_freqs)
fm.fit(frequency, interp_spectrum, freq_range=(start_freq, end_freq))

# generate plot
fm.plot()
mpl_fig = plt.gcf()
plotly_fig = tls.mpl_to_plotly(mpl_fig)
json_fig = pio.to_json(plotly_fig)

# extract summary parameters
aperiodic_params = fm.get_params('aperiodic')
summary_params = {
    "aperiodic_params": aperiodic_params,
    "periodic_params": fm.get_params('periodic'),
    "quality_metrics": np.array([fm.get_metrics('error_mae'), fm.get_metrics('gof_rsquared')])
}

if aperiodic_mode == 'fixed':
    offset, exponent = aperiodic_params
    aperiodic_fit = 10**(offset - exponent * np.log10(bounded_frequency))
elif aperiodic_mode == 'knee':
    offset, knee, exponent = aperiodic_params
    aperiodic_fit = 10**(offset - np.log10(knee + bounded_frequency ** exponent))
else:
    raise ValueError(f"Invalid aperiodic mode: {aperiodic_mode}")

# find oscillatory activity relative to aperiodic fit
interp_spectrum = interp_spectrum[np.isin(frequency, bounded_frequency)]  # restrict to bounded frequency
oscillatory_activity = interp_spectrum - aperiodic_fit

# fetch bosc parameters
dt, detection_thresh = (analysis.FBOSCParamset & key).fetch1(
    "dt", "detection_thresh")

# get chi-square factor for thresholding
chi2_factor = chi2.ppf(detection_thresh, df=2) / 2

# loop through time bins and perform fBOSC analysis
time_bins = np.arange(0, time[-1], dt)
epoch_data = {
    **{f"{band_name}_times": [] for band_name in frequency_band_masks.keys()},
    **{f"{band_name}_heights": [] for band_name in frequency_band_masks.keys()},
    "aperiodic_offset": [],
    "aperiodic_knee": [],
    "aperiodic_exponent": [],
    "mae": [],
    "r_squared": [],
    }
for t_start in time_bins:

    # get spectrum within time bin
    epoch_spectrum = np.mean(mean_spectrum[:, (t_start <= time) & (time < t_start + dt)], axis=1)

    # interpolate mean_spectrum to account for 60 Hz line noise removal
    interp_epoch_spectrum = interpolate_spectrum(frequency, epoch_spectrum, notch_freqs)

    # fit model
    fm.fit(frequency, interp_epoch_spectrum, freq_range=(start_freq, end_freq))
    interp_epoch_spectrum = interp_epoch_spectrum[np.isin(frequency, bounded_frequency)]  # restrict to bounded frequency

    # extract aperiodic fit parameters
    aperiodic_params = fm.get_params('aperiodic')

    # get aperiodic fit
    if aperiodic_mode == 'fixed':
        offset, exponent = aperiodic_params
        aperiodic_fit = 10**(offset - exponent * np.log10(bounded_frequency))
    elif aperiodic_mode == 'knee':
        offset, knee, exponent = aperiodic_params
        aperiodic_fit = 10**(offset - np.log10(knee + bounded_frequency ** exponent))
    else:
        raise ValueError(f"Invalid aperiodic mode: {aperiodic_mode}")
    
    # extract aperiodic metrics
    epoch_data["aperiodic_offset"].append(offset)
    epoch_data["aperiodic_knee"].append(knee if aperiodic_mode == 'knee' else 0)
    epoch_data["aperiodic_exponent"].append(exponent)
    
    # get chi-square threshold for burst detection
    threshold_spectrum = aperiodic_fit * chi2_factor

    # extract if specific frequency bands have bursts (spectral power > threshold)
    for band_name, band_mask in frequency_band_masks.items():
        if np.any(interp_epoch_spectrum[band_mask] > threshold_spectrum[band_mask]):
            epoch_data[f"{band_name}_times"].append(t_start + dt/2) # store center time of bin
            epoch_data[f"{band_name}_heights"].append(np.max(interp_epoch_spectrum[band_mask] - aperiodic_fit[band_mask])) # store max height above aperiodic fit

    # extract fit metrics
    epoch_data["mae"].append(fm.get_metrics('error_mae'))
    epoch_data["r_squared"].append(fm.get_metrics('gof_rsquared'))

# convert lists to arrays
for key_name in epoch_data.keys():
    epoch_data[key_name] = np.array(epoch_data[key_name])

[2026-03-30 18:28:01,249][WARNING]: Reconnecting to MySQL server.
c:\Users\Organoid PC\anaconda3\envs\utah_organoids\lib\site-packages\plotly\matplotlylib\renderer.py:609: UserWarning:

I found a path object that I don't think is part of a bar chart. Ignoring.



In [28]:
fm.get_params('periodic')

array([[ 9.19 ,  0.331, 12.   ],
       [26.579,  0.288, 12.   ]])

In [29]:
fm.get_metrics(category='gof_rsquared')

0.9901608142204062

In [31]:
fm.get_params("all")

AttributeError: 'ModelParameters' object has no attribute 'all'